Исходные данные и параметры:

In [3]:
import numpy as np

sites = [
    "GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA",
    "ACAGTCAGC", "TAGGTCAGC", "CAGGTCAGC",
    "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC",
    "CAGGTTGGC"
]

background = {'A': 0.295, 'T': 0.295, 'G': 0.205, 'C': 0.205}
nucleotides_order = ['A', 'T', 'G', 'C'] # порядок строк в матрице!!!
n_sites = len(sites)
seq_length = len(sites[0])

Функция для получения PFM (Position Frequency Matrix)

In [4]:
def get_pfm(sites_list, nuc_order):
    """Возвращает матрицу PFM 4xL (количество вхождений нуклеотидов по позициям)."""
    pfm = np.zeros((len(nuc_order), seq_length), dtype=int)
    for i, nuc in enumerate(nuc_order):
        for pos in range(seq_length):
            count = sum(1 for seq in sites_list if seq[pos] == nuc)
            pfm[i, pos] = count
    return pfm

In [5]:
def pfm_to_ppm(pfm_matrix, pseudocount=0.1):
    """Преобразует PFM в PPM (частоты с псевдосчётом)."""
    pfm_pseudo = pfm_matrix + pseudocount
    column_sums = pfm_pseudo.sum(axis=0)
    ppm = pfm_pseudo / column_sums
    return ppm

In [6]:
def ppm_to_pwm(ppm_matrix, bg_dict, nuc_order):
    """Преобразует PPM в PWM (веса как log2(частота / фон))."""
    bg_vector = np.array([bg_dict[nuc] for nuc in nuc_order]).reshape(-1, 1)
    # Добавляем небольшое число, чтобы избежать log(0), но в PPM у нас нет нулей из-за псевдосчёта
    pwm = np.log2(ppm_matrix / bg_vector)
    return pwm

In [7]:
pfm = get_pfm(sites, nucleotides_order)
ppm = pfm_to_ppm(pfm, pseudocount=0.1)
pwm = ppm_to_pwm(ppm, background, nucleotides_order)

print("PFM(сырой счёт)")
print(pfm)
print("\n PPM (частоты с псевдосчётом 0.1)")
print(np.round(ppm, 4))
print("\n PWM (веса log2) ")
print(np.round(pwm, 4))

max_score = 0
min_score = 0
max_seq = ""
min_seq = ""

# просто перебором решаем задачу
for pos in range(seq_length):
    max_nuc_idx = np.argmax(pwm[:, pos])
    max_score += pwm[max_nuc_idx, pos]
    max_seq += nucleotides_order[max_nuc_idx]

    min_nuc_idx = np.argmin(pwm[:, pos])
    min_score += pwm[min_nuc_idx, pos]
    min_seq += nucleotides_order[min_nuc_idx]

print(f"Максимальный скор: {max_score:.4f}")
print(f"Последовательность для максимума: {max_seq}")
print(f"Минимальный скор: {min_score:.4f}")
print(f"Последовательность для минимума: {min_seq}")

PFM(сырой счёт)
[[ 1  8  1  0  0  2  7  2  1]
 [ 2  0  0  0 10  2  0  0  1]
 [ 1  0  8 10  0  0  3  8  0]
 [ 6  2  1  0  0  6  0  0  8]]

 PPM (частоты с псевдосчётом 0.1)
[[0.1058 0.7788 0.1058 0.0096 0.0096 0.2019 0.6827 0.2019 0.1058]
 [0.2019 0.0096 0.0096 0.0096 0.9712 0.2019 0.0096 0.0096 0.1058]
 [0.1058 0.0096 0.7788 0.9712 0.0096 0.0096 0.2981 0.7788 0.0096]
 [0.5865 0.2019 0.1058 0.0096 0.0096 0.5865 0.0096 0.0096 0.7788]]

 PWM (веса log2) 
[[-1.4798  1.4006 -1.4798 -4.9392 -4.9392 -0.5469  1.2105 -0.5469 -1.4798]
 [-0.5469 -4.9392 -4.9392 -4.9392  1.719  -0.5469 -4.9392 -4.9392 -1.4798]
 [-0.9547 -4.4141  1.9257  2.2441 -4.4141 -4.4141  0.5401  1.9257 -4.4141]
 [ 1.5166 -0.0218 -0.9547 -4.4141 -4.4141  1.5166 -4.4141 -4.4141  1.9257]]
Максимальный скор: 15.3846
Последовательность для максимума: CAGGTCAGC
Минимальный скор: -39.9434
Последовательность для минимума: ATTAAGTTG
